In [ ]:
# ==========================================
# SCENARIO 1: RETAIL INVENTORY ANALYSIS
# ==========================================

#  Prompt:
"""
Act as a Senior Retail Supply Chain Data Scientist. I have a pandas DataFrame containing inventory data with the following columns: product_id, category, stock_level, last_restock_date, sales_last_30_days, supplier_lead_time, and unit_cost. 

Please write a Python script to:
1. Calculate inventory turnover rates.
2. Identify slow-moving items.
3. Predict potential stockouts by comparing current stock and sales velocity against the supplier lead time.
4. Create visualizations to display these insights.

Ensure you include robust data validation (especially for date parsing and zero-division errors) and document your functions clearly.
"""

# Generated Code:
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_inventory(df):
    # Data Validation and Date Handling
    try:
        df['last_restock_date'] = pd.to_datetime(df['last_restock_date'])
    except Exception as e:
        print(f"Error parsing dates: {e}")
        return None

    # 1. Inventory Turnover Rate (Sales / Average Stock)
    # Adding a small constant to avoid division by zero
    df['turnover_rate'] = df['sales_last_30_days'] / (df['stock_level'] + 1e-5)

    # 2. Identify Slow-Moving Items (Low turnover and high stock)
    threshold = df['turnover_rate'].quantile(0.25)
    df['is_slow_moving'] = (df['turnover_rate'] < threshold) & (df['stock_level'] > 0)

    # 3. Predict Potential Stockouts
    # Daily sales velocity
    df['daily_sales'] = df['sales_last_30_days'] / 30
    df['days_of_inventory'] = df['stock_level'] / (df['daily_sales'] + 1e-5)
    df['potential_stockout'] = df['days_of_inventory'] <= df['supplier_lead_time']

    return df

def visualize_inventory(df):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='stock_level', y='sales_last_30_days', hue='is_slow_moving')
    plt.title('Stock Level vs Sales (Slow Moving Highlighted)')
    plt.show()
"""

# Follow-up Prompts:
"""
The code is good, but the visualization is a bit basic and the error handling only covers date parsing. Could you optimize the code to include `try-except` blocks around the calculations to prevent zero-division warnings natively, and add a second visualization showing stockout risks by category?
"""

# Final Solution:
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_inventory_optimized(df):
    # Data Validation and Error Handling
    if df.empty:
        raise ValueError("The provided DataFrame is empty.")
        
    try:
        df = df.copy()
        df['last_restock_date'] = pd.to_datetime(df['last_restock_date'])
        
        # 1. Turnover Rate (Safely handling division by zero)
        df['turnover_rate'] = np.where(df['stock_level'] > 0, 
                                       df['sales_last_30_days'] / df['stock_level'], 
                                       0)
                                       
        # 2. Slow-Moving Items
        turnover_threshold = df['turnover_rate'].quantile(0.25)
        df['is_slow_moving'] = (df['turnover_rate'] <= turnover_threshold) & (df['stock_level'] > 0)
        
        # 3. Stockout Prediction
        df['daily_sales'] = df['sales_last_30_days'] / 30
        # Days until out of stock
        df['days_until_empty'] = np.where(df['daily_sales'] > 0,
                                          df['stock_level'] / df['daily_sales'],
                                          np.inf)
        
        df['stockout_risk'] = df['days_until_empty'] <= df['supplier_lead_time']
        
        return df
        
    except KeyError as e:
        print(f"Missing required column: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def plot_inventory_insights(df):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Slow Moving Items
    sns.scatterplot(data=df, x='stock_level', y='turnover_rate', hue='is_slow_moving', ax=axes[0])
    axes[0].set_title('Inventory Turnover vs Stock Level')
    
    # Plot 2: Stockout Risks by Category
    risk_counts = df.groupby('category')['stockout_risk'].sum().reset_index()
    sns.barplot(data=risk_counts, x='category', y='stockout_risk', ax=axes[1], palette='Reds_r')
    axes[1].set_title('Number of Items at Risk of Stockout by Category')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
"""

In [ ]:
# ==========================================
# SCENARIO 2: WEBSITE ANALYTICS DEBUG
# ==========================================

# Prompt:
"""
Act as a Senior QA Data Engineer. The marketing team reported that our `analyze_user_engagement` function is returning impossible results: bounce rates over 100% and negative session times. 

Here is the buggy code:
def analyze_user_engagement(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }
    device_metrics = logs_df.groupby('device_type').agg({'session_id': 'count', 'duration': 'mean', 'page_views': 'sum'})
    return metrics, device_metrics

Please debug these calculations. Specifically:
1. Add data validation to clean out negative times or invalid page views.
2. Implement proper mathematical logic for the bounce rate and time calculations.
3. Create a summary visualization for the device metrics using seaborn or matplotlib.

Crucially, explain exactly WHY the original code produced >100% bounce rates and negative times, and how your fix resolves this.
"""

# Generated Code:
"""
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Explanation of Original Errors:
# 1. Negative Session Times: The original code used `.sum()` on duration without filtering out corrupted negative values from the logs.
# 2. >100% Bounce Rate: The lambda function `apply(lambda x: x == 1).mean()` was evaluating page views per row, not aggregating the total views per session first. If summed incorrectly, this breaks the 0-1 percentage bound. 

def analyze_user_engagement(logs_df):
    # 1. Data Validation: Filter out negative durations and zero/negative page views
    clean_df = logs_df[(logs_df['duration'] >= 0) & (logs_df['page_views'] > 0)].copy()
    
    # 2. Session-level aggregation
    session_agg = clean_df.groupby('session_id').agg({
        'page_views': 'sum',
        'duration': 'sum',
        'device_type': 'first' # Assuming device doesn't change mid-session
    })
    
    # Correct Metrics Calculation
    total_sessions = len(session_agg)
    bounced_sessions = len(session_agg[session_agg['page_views'] == 1])
    
    metrics = {
        'bounce_rate': bounced_sessions / total_sessions if total_sessions > 0 else 0,
        'avg_session_time': session_agg['duration'].mean(),
        'pages_per_session': session_agg['page_views'].mean()
    }
    
    # Device Metrics
    device_metrics = session_agg.groupby('device_type').agg({
        'page_views': 'count', # Counts unique sessions per device
        'duration': 'mean',
        'page_views': 'mean'
    }).rename(columns={'page_views': 'avg_page_views'})
    
    return metrics, device_metrics

def visualize_device_metrics(device_metrics):
    device_metrics.reset_index(inplace=True)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=device_metrics, x='device_type', y='duration')
    plt.title('Average Session Duration by Device Type')
    plt.ylabel('Duration (seconds)')
    plt.show()
"""

# Follow-up Prompts:
"""
This is a great fix. However, the visualization function only plots duration. Can you optimize the visualization function to plot both Average Session Duration AND Average Page Views side-by-side using subplots? Also, add a quick print statement in the validation step to warn the user how many corrupted rows were dropped.
"""

# Final Solution:
"""
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_user_engagement_optimized(logs_df):
    initial_rows = len(logs_df)
    
    # Data Validation
    clean_df = logs_df[(logs_df['duration'] >= 0) & (logs_df['page_views'] > 0)].copy()
    dropped_rows = initial_rows - len(clean_df)
    
    if dropped_rows > 0:
        print(f"WARNING: Dropped {dropped_rows} corrupted rows containing negative durations or zero page views.")
        
    if clean_df.empty:
        return None, None

    # Session-level aggregation
    session_agg = clean_df.groupby('session_id').agg({
        'page_views': 'sum',
        'duration': 'sum',
        'device_type': 'first' 
    })
    
    total_sessions = len(session_agg)
    bounced_sessions = len(session_agg[session_agg['page_views'] == 1])
    
    metrics = {
        'bounce_rate': bounced_sessions / total_sessions if total_sessions > 0 else 0,
        'avg_session_time': session_agg['duration'].mean(),
        'pages_per_session': session_agg['page_views'].mean()
    }
    
    device_metrics = session_agg.groupby('device_type').agg({
        'duration': 'mean',
        'page_views': 'mean'
    }).rename(columns={'duration': 'avg_duration', 'page_views': 'avg_page_views'}).reset_index()
    
    return metrics, device_metrics

def visualize_device_metrics_optimized(device_metrics):
    if device_metrics is None or device_metrics.empty:
        print("No data available to visualize.")
        return
        
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Average Duration
    sns.barplot(data=device_metrics, x='device_type', y='avg_duration', ax=axes[0], palette='Blues_d')
    axes[0].set_title('Average Session Duration by Device Type')
    axes[0].set_ylabel('Duration (Seconds)')
    
    # Plot 2: Average Page Views
    sns.barplot(data=device_metrics, x='device_type', y='avg_page_views', ax=axes[1], palette='Greens_d')
    axes[1].set_title('Average Page Views by Device Type')
    axes[1].set_ylabel('Page Views')
    
    plt.tight_layout()
    plt.show()
"""

In [ ]:
# ==========================================
# SCENARIO 3: CUSTOMER SEGMENTATION QUERY
# ==========================================

#  Prompt:
"""
Act as a Lead Database Architect. I need an optimized PostgreSQL query to segment customers based on their purchasing behavior. 

Database Schema:
- user_activity: user_id, last_login_date, feature_usage_count, account_type
- transactions: transaction_id, user_id, transaction_date, amount, platform
- user_preferences: user_id, communication_preference, interface_theme, notification_settings

Task:
1. Filter for 'Active' users (users who have logged in within the last 30 days).
2. Identify high-value customers (specifically, the top 20% of active users by total spending amount).
3. Aggregate the user preference trends (e.g., count of most popular interface_themes) for these specific high-value customers.

Please write the query using Common Table Expressions (CTEs) for readability, and briefly explain how the percentile calculation handles the top 20% constraint.
"""

# Generated Code:
"""
/*
Explanation of Percentile Logic:
We use the PERCENT_RANK() window function ordered by total_spending descending. 
This assigns a relative rank between 0 and 1 to each user. Filtering by `<= 0.20` captures the top 20% of spenders.
*/

WITH ActiveUsers AS (
    SELECT user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),
TotalSpend AS (
    SELECT 
        t.user_id,
        SUM(t.amount) AS total_spending
    FROM transactions t
    INNER JOIN ActiveUsers au ON t.user_id = au.user_id
    GROUP BY t.user_id
),
RankedSpenders AS (
    SELECT 
        user_id,
        total_spending,
        PERCENT_RANK() OVER (ORDER BY total_spending DESC) AS spend_percentile
    FROM TotalSpend
),
HighValueUsers AS (
    SELECT user_id
    FROM RankedSpenders
    WHERE spend_percentile <= 0.20
)
SELECT 
    up.interface_theme,
    up.communication_preference,
    COUNT(*) AS user_count
FROM user_preferences up
INNER JOIN HighValueUsers hvu ON up.user_id = hvu.user_id
GROUP BY 
    up.interface_theme,
    up.communication_preference
ORDER BY 
    user_count DESC;
"""

# Follow-up Prompts:
"""
The query structure is excellent and highly readable. However, using `PERCENT_RANK()` can sometimes be computationally heavy on massive datasets. Could you provide a slightly more optimized version using `NTILE(5)` to bucket the users into quintiles instead? Please explain how the query optimizer treats this differently.
"""

# Final Solution:
"""
/*
Optimization Explanation:
Using `NTILE(5)` divides the partition into 5 equal buckets (quintiles). 
Filtering for `NTILE(5) = 1` immediately yields the top 20%. This is often more efficient for the query planner than calculating exact floating-point percentiles across millions of rows with `PERCENT_RANK()`, as it avoids granular rank calculation in favor of simple bucketing.
*/

WITH ActiveUsers AS (
    SELECT user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),
TotalSpend AS (
    SELECT 
        t.user_id,
        SUM(t.amount) AS total_spending
    FROM transactions t
    INNER JOIN ActiveUsers au ON t.user_id = au.user_id
    GROUP BY t.user_id
),
QuintileSpenders AS (
    SELECT 
        user_id,
        NTILE(5) OVER (ORDER BY total_spending DESC) AS spend_bucket
    FROM TotalSpend
),
HighValueUsers AS (
    -- Bucket 1 represents the top 20% (1/5th) of spenders
    SELECT user_id
    FROM QuintileSpenders
    WHERE spend_bucket = 1
)
SELECT 
    up.interface_theme,
    up.communication_preference,
    COUNT(*) AS user_count
FROM user_preferences up
INNER JOIN HighValueUsers hvu ON up.user_id = hvu.user_id
GROUP BY 
    up.interface_theme,
    up.communication_preference
ORDER BY 
    user_count DESC;
"""